In [11]:
import random
import time
import numpy as np

In [12]:
class node:
    """
    Clase para representar los nodos del árbol.
    """

    def __init__(self, tablero, turno, g_p):
        #estructuras de navegación
        self.pad=None            # referencia al nodo padre
        self.hijos = []          # lista de referencias a los nodos hijos

        #datos contenida
        self.tablero = tablero   # tablero del juego específico en el nodo
        self.g_p = g_p           # 1: ganador, -1: perdedor, 0: sigue el juego
        self.uct = 6.0           # calificación asignada al tablero
        self.n = 0               # cantidad de veces que se ha visitado el nodo
        self.wins = 0            # cantidad de victorias del subárbol
        self.turno = turno       # turno del siguiente tiro

In [ ]:
class hexapawn:
    def __init__(self):
        self.seed = int(time.time_ns())
        self.gen = random.Random(self.seed)

    #Regresa un número entero aleatorio
    def random_int(self):
        return self.gen.randint(1, 1000)
    
    #Coloca el tablero para un juego desde 0
    def poner_el_juego(self, n, turno=0):
        '''
        pone el juego desde el inicio, donde el tablero es siempre cuadrado, y la cantidad
        de piezas dependen del tamaño del tablero
        Entrada:
        int n: tamaño del tablero
        int turno=0: turno actual (preset a cero, o sea turno humano)
        
        Salida:
        salida: tablero
        '''
        primera_fila=[2 for i in range(n)]
        ultima_fila=[1 for i in range(n)]
        fila_medio=[0 for i in range(n)]

        tablero=[]
        tablero.append(primera_fila.copy())
        for i in range(n-2): tablero.append(fila_medio.copy())
        tablero.append(ultima_fila.copy())

        return tablero, 1 if turno==0 or turno==1 else 2
    
    #Imprime de manera bonita el estado del juego, junto con el turno
    def ver_tablero(self, tablero, turno=1):
        count=0
        print("╔", end="")
        for i in range(len(tablero)*4-1):
            print("═", end="")
            count+=1
            if(count==100):
                return 
            
        print("╗   filas     turno de: ", "X" if turno==1 else "O")
        
        for i in range(len(tablero)):
            print("║", end="")

            for j in range(len(tablero[1])):
                print("   " if tablero[i][j]==0 else " X " if tablero[i][j]==1 else " O ", end="")
                if j!=len(tablero[1])-1: print("│", end="")
                
            print("║ ", i)
            
            if i!=len(tablero):
                print("║", end="")
                for j in range(len(tablero[1])):
                    for k in range(3):
                        print("-", end="")
                        
                    if j!=len(tablero[1])-1:
                        print("┼", end="")
                print("║")
        
        print("╚", end="")
        for i in range(len(tablero[1])*4-1):
            print("═", end="")

        print("╝")
        for i in range(len(tablero[1])):
            print(" ", i, " ", end="")
            
        print("\n\ncolumnas\n\n")
        
    #Cambia de turno para el siguiente
    def siguiente_turno(turno):
        return 2 if turno==2 else 1
    
    
    #Filas original, columnas original, filas terminal, columnas terminal
    #te dice si el tiro que quieres hacer es legal o no
    def tiro_legal(f_o, c_o, f_t, c_t, tablero):
        if(f_t<0 or f_t>len(tablero)-1 or c_t<0 or c_t>len(tablero)-1):
            return False
        
        if(tablero[f_o][c_o]==1):
            return (((f_t==f_o-1) and (c_o==c_t)) and (tablero[f_t][c_t]==0)) or (((f_t==f_o-1) and (c_o==c_t-1)) and (tablero[f_t][c_t]==2)) or (((f_t==f_o-1) and (c_o==c_t+1)) and (tablero[f_t][c_t]==2))
        
        elif(tablero[f_o][c_o]==2):
            return (((f_t==f_o+1) and (c_o==c_t)) and (tablero[f_t][c_t]==0)) or (((f_t==f_o+1) and (c_o==c_t-1)) and (tablero[f_t][c_t]==1)) or (((f_t==f_o+1) and (c_o==c_t+1)) and (tablero[f_t][c_t]==1));
        
        else:
            return False
        
    
    #recibe un tablero junto con el turno y da un tiro aleatorio, regresa el tablero con el
    #tiro regristrado y el siguiente turno en una tupla
    def tiro_random(self, tablero, turno):
        #posibles almacena un vector, de coordenadas de todas las fichas del jugador del que queremos hacer el tiro
        posibles=[]

        for i in range(len(tablero)):
            for j in range(len(tablero[0])):
                if(tablero[i][j]==turno):
                    posibles.append([i, j])

        self.gen.shuffle(posibles)
        
        while posibles:
            direcciones=[-1,0,1]
            elegido=posibles[len(posibles)-1]
            self.gen.shuffle(direcciones)
            
            while direcciones:
                if(self.tiro_legal(elegido[0],
                                   elegido[1],
                                   elegido[0]+(1 if turno==1 else -1),
                                   elegido[1]+direcciones[len(direcciones)-1],
                                   tablero)):
                    tablero[elegido[0]][elegido[1]]=0
                    tablero[elegido[0]-1 if turno==1 else elegido[0]+1][elegido[1]+direcciones[len(direcciones)-1]]=turno

                    return (tablero, 2 if turno==1 else 1)
                
                else:
                    direcciones.pop()
                
            
            posibles.pop()
        
        return tablero, 0
    
    

In [ ]:
juego=hexapawn()

tablero, turno=juego.poner_el_juego(10, 0)
juego.ver_tablero(tablero, turno)

╔═══════════════════════════════════════╗   filas     turno de:  X
║ O │ O │ O │ O │ O │ O │ O │ O │ O │ O ║  0
║---┼---┼---┼---┼---┼---┼---┼---┼---┼---║
║   │   │   │   │   │   │   │   │   │   ║  1
║---┼---┼---┼---┼---┼---┼---┼---┼---┼---║
║   │   │   │   │   │   │   │   │   │   ║  2
║---┼---┼---┼---┼---┼---┼---┼---┼---┼---║
║   │   │   │   │   │   │   │   │   │   ║  3
║---┼---┼---┼---┼---┼---┼---┼---┼---┼---║
║   │   │   │   │   │   │   │   │   │   ║  4
║---┼---┼---┼---┼---┼---┼---┼---┼---┼---║
║   │   │   │   │   │   │   │   │   │   ║  5
║---┼---┼---┼---┼---┼---┼---┼---┼---┼---║
║   │   │   │   │   │   │   │   │   │   ║  6
║---┼---┼---┼---┼---┼---┼---┼---┼---┼---║
║   │   │   │   │   │   │   │   │   │   ║  7
║---┼---┼---┼---┼---┼---┼---┼---┼---┼---║
║   │   │   │   │   │   │   │   │   │   ║  8
║---┼---┼---┼---┼---┼---┼---┼---┼---┼---║
║ X │ X │ X │ X │ X │ X │ X │ X │ X │ X ║  9
║---┼---┼---┼---┼---┼---┼---┼---┼---┼---║
╚═══════════════════════════════════════╝
  0    1    2    3   